# Car Plate Detection & OCR - Testing & Visualization Guide

This notebook demonstrates how to:
- ✅ Test the plate detection module
- ✅ Test the OCR module
- ✅ Visualize detections and results
- ✅ Check confidence scores and statistics
- ✅ Debug errors and issues

In [ ]:
# Step 1: Import Required Libraries
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import logging
from typing import Tuple, List
import time

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

print("✅ All libraries imported successfully")

## Step 2: Load Your Custom Modules

Add the src folder to Python path and import your modules

In [ ]:
# Add src folder to path
src_path = Path("src")
if src_path.exists():
    sys.path.insert(0, str(src_path.absolute()))
    print(f"✅ Added to path: {src_path.absolute()}")
else:
    print(f"⚠️  Warning: src folder not found at {src_path.absolute()}")

# Import custom modules
try:
    from plate_detector import PlateDetector
    from ocr import PlateOCR
    print("✅ Successfully imported PlateDetector and PlateOCR")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure plate_detector.py and ocr.py are in the src folder")

## Step 3: Create Test Image

Generate a sample image or load from file for testing

In [ ]:
def load_image(image_path: str) -> np.ndarray:
    """Load image from file"""
    if not os.path.exists(image_path):
        print(f"❌ Image not found: {image_path}")
        return None
    
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Failed to load image: {image_path}")
        return None
    
    print(f"✅ Loaded image: {image_path}")
    print(f"   Shape: {image.shape}, Size: {image.nbytes / 1024:.2f} KB")
    return image

def create_dummy_image(height: int = 400, width: int = 600) -> np.ndarray:
    """Create a dummy blue image for testing"""
    image = np.ones((height, width, 3), dtype=np.uint8) * 100
    
    # Add some text to make it look like a plate detection scenario
    cv2.putText(image, "Test Image for Detection", (50, 150),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 0), 2)
    
    # Add a fake plate region
    cv2.rectangle(image, (100, 200), (500, 300), (0, 255, 0), 2)
    cv2.putText(image, "ABC-1234", (150, 260),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 2)
    
    print(f"✅ Created dummy test image: {image.shape}")
    return image

# Try to load from detection_results.csv or create dummy image
test_image_path = "test_image.jpg"
if os.path.exists(test_image_path):
    test_image = load_image(test_image_path)
else:
    print("⚠️  No test_image.jpg found, creating dummy image for demonstration")
    test_image = create_dummy_image()

## Step 4: Visualization Utilities

Helper functions to display images and results

In [ ]:
def show_image(image: np.ndarray, title: str = "Image", figsize: Tuple = (12, 8)):
    """Display image using matplotlib"""
    if image is None:
        print("❌ Cannot display None image")
        return
    
    plt.figure(figsize=figsize)
    
    # Convert BGR to RGB for display
    if len(image.shape) == 3 and image.shape[2] == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    
    plt.imshow(image_rgb)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_images_grid(images: List[Tuple[np.ndarray, str]], rows: int = 1, figsize: Tuple = (16, 5)):
    """Display multiple images in a grid"""
    if not images:
        print("❌ No images to display")
        return
    
    cols = len(images)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    
    if cols == 1:
        axes = [axes]
    
    for idx, (image, title) in enumerate(images):
        ax = axes[idx] if cols > 1 else axes[0]
        
        if len(image.shape) == 3 and image.shape[2] == 3:
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        else:
            image_rgb = image
        
        ax.imshow(image_rgb)
        ax.set_title(title, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

def show_detections_with_boxes(image: np.ndarray, detections: dict, title: str = "Detection Results"):
    """Display image with detection boxes"""
    if image is None or detections['num_detections'] == 0:
        print("⚠️  No detections to display")
        show_image(image, title)
        return
    
    # Draw boxes on image
    output = image.copy()
    height, width = image.shape[:2]
    
    for box, score in zip(detections['detection_boxes'], detections['detection_scores']):
        ymin, xmin, ymax, xmax = box
        x_min = int(xmin * width)
        x_max = int(xmax * width)
        y_min = int(ymin * height)
        y_max = int(ymax * height)
        
        cv2.rectangle(output, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        label = f"{score:.3f}"
        cv2.putText(output, label, (x_min, y_min - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    show_image(output, f"{title} ({detections['num_detections']} detections)")

print("✅ Visualization utilities defined")

## Step 5: Initialize Plate Detector

⚠️ **IMPORTANT**: You need a pre-trained model file for this to work!

In [ ]:
# Configure detector settings
MODEL_PATH = "path/to/your/saved_model"  # Update this path!
CONFIDENCE_THRESHOLD = 0.5
NMS_THRESHOLD = 0.5
PADDING = 10

# Check if model exists
if os.path.exists(MODEL_PATH):
    try:
        detector = PlateDetector(
            model_path=MODEL_PATH,
            confidence_threshold=CONFIDENCE_THRESHOLD,
            nms_threshold=NMS_THRESHOLD,
            padding=PADDING
        )
        print("✅ PlateDetector initialized successfully")
        print(f"   Configuration:")
        print(f"   - Confidence Threshold: {CONFIDENCE_THRESHOLD}")
        print(f"   - NMS Threshold: {NMS_THRESHOLD}")
        print(f"   - Padding: {PADDING} pixels")
    except Exception as e:
        print(f"❌ Failed to initialize detector: {e}")
        detector = None
else:
    print(f"⚠️  Model not found at: {MODEL_PATH}")
    print("📝 Steps to fix:")
    print("   1. Download or train a TensorFlow object detection model")
    print("   2. Update MODEL_PATH variable above")
    print("   3. Ensure model is in SavedModel format")
    detector = None

## Step 6: Test Plate Detection

Run detection on test image and visualize results

In [ ]:
if detector and test_image is not None:
    print("🔍 Running detection on test image...")
    print(f"   Image size: {test_image.shape}")
    
    # Measure detection time
    start_time = time.time()
    detections = detector.detect(test_image)
    detection_time = time.time() - start_time
    
    print(f"\n✅ Detection completed in {detection_time:.3f} seconds")
    print(f"   Detections found: {detections['num_detections']}")
    
    if detections['num_detections'] > 0:
        print(f"\n📊 Detection Statistics:")
        print(f"   Plates detected: {detections['num_detections']}")
        
        # Print confidence scores
        scores = detections['detection_scores']
        print(f"   Confidence scores: Min={scores.min():.3f}, Max={scores.max():.3f}, Avg={scores.mean():.3f}")
        
        # Print details for each detection
        for idx, (score, class_id) in enumerate(zip(scores, detections['detection_classes'])):
            print(f"   Plate {idx+1}: Score={score:.3f}, Class={int(class_id)}")
        
        # Visualize detections
        viz_image = detector.visualize(test_image, detections)
        show_image(viz_image, f"Detection Results ({detections['num_detections']} plates)")
        
        # Get stats
        stats = detector.get_detection_stats()
        print(f"\n📈 Detector Statistics:")
        print(f"   Total detections in session: {stats['total_detections']}")
    else:
        print("⚠️  No plates detected in the image")
else:
    print("❌ Cannot run detection: detector or image not available")
    print("   Please ensure:")
    print("   1. Model path is correctly set")
    print("   2. Test image is loaded")

## Step 7: Initialize & Test OCR

Initialize the OCR module and extract text from plate regions

In [ ]:
try:
    print("🚀 Initializing OCR module...")
    ocr = PlateOCR(languages=['en'], use_gpu=None)
    print("✅ OCR initialized successfully")
    
    # Get OCR stats
    stats = ocr.get_ocr_stats()
    print(f"   Languages: {stats['languages']}")
    print(f"   Using GPU: {stats['using_gpu']}")
    print(f"   Reader available: {stats['reader_available']}")
    
except Exception as e:
    print(f"❌ Failed to initialize OCR: {e}")
    ocr = None

### Extract Text from Detected Plates

If detection was successful, extract text from each detected plate region

In [ ]:
if detector and ocr and test_image is not None and 'detections' in locals():
    if detections['num_detections'] > 0:
        print(f"📖 Running OCR on {detections['num_detections']} detected plates...\n")
        
        # Get plate regions
        plate_regions = detector.get_plate_regions(test_image, detections)
        print(f"✅ Extracted {len(plate_regions)} plate regions")
        
        if plate_regions:
            all_results = []
            
            for idx, (plate_img, (x_min, y_min, x_max, y_max)) in enumerate(plate_regions):
                print(f"\n--- Plate {idx+1} ---")
                print(f"Position: ({x_min}, {y_min}) to ({x_max}, {y_max})")
                print(f"Size: {plate_img.shape}")
                
                # Measure OCR time
                start_time = time.time()
                ocr_result, viz = ocr.extract_with_visualization(plate_img)
                ocr_time = time.time() - start_time
                
                print(f"OCR Time: {ocr_time:.3f}s")
                print(f"Raw text: '{ocr_result['full_text']}'")
                print(f"Cleaned text: '{ocr_result['cleaned_text']}'")
                print(f"Average confidence: {ocr_result['avg_confidence']:.3f}")
                print(f"Valid plate: {ocr_result['is_valid_plate']} ✓" if ocr_result['is_valid_plate'] else f"Valid plate: {ocr_result['is_valid_plate']} ✗")
                print(f"Character count: {len(ocr_result['text'])}")
                
                # Display individual confidence scores
                if ocr_result['text']:
                    print(f"Individual confidence scores:")
                    for char, conf in zip(ocr_result['text'], ocr_result['confidence']):
                        print(f"  '{char}': {conf:.3f}")
                
                all_results.append(ocr_result)
                
                # Show visualization
                show_image(viz, f"OCR Result - Plate {idx+1}: {ocr_result['cleaned_text']}")
            
            # Summary
            print(f"\n{'='*50}")
            print(f"📊 OCR SUMMARY")
            print(f"{'='*50}")
            print(f"Total plates processed: {len(all_results)}")
            print(f"Valid plates detected: {sum(1 for r in all_results if r['is_valid_plate'])}")
            
            print(f"\nExtracted Plate Numbers:")
            for idx, result in enumerate(all_results, 1):
                text = result['cleaned_text'] if result['cleaned_text'] else result['full_text']
                print(f"  {idx}. {text} (confidence: {result['avg_confidence']:.3f})")
else:
    print("⚠️  Cannot run OCR: missing detector, OCR module, or detection results")

## Step 8: Error Handling & Debugging

Test error cases and debug issues

In [ ]:
print("🧪 Testing Error Handling\n")

# Test 1: Empty image
print("Test 1: Empty/None image")
try:
    if detector:
        result = detector.detect(None)
        print(f"   Result: {result}")
        print(f"   Detections: {result['num_detections']}")
        print("   ✅ Handled gracefully")
except Exception as e:
    print(f"   ❌ Error: {e}")

# Test 2: Invalid image shape
print("\nTest 2: Invalid image shape")
try:
    invalid_img = np.array([1, 2, 3])  # 1D array
    if detector:
        result = detector.detect(invalid_img)
        print(f"   ❌ Should have raised error")
except ValueError as e:
    print(f"   ✅ Caught error as expected: {e}")
except Exception as e:
    print(f"   ❌ Unexpected error: {e}")

# Test 3: OCR with empty plate image
print("\nTest 3: OCR with empty plate image")
try:
    if ocr:
        empty_plate = np.zeros((30, 100, 3), dtype=np.uint8)
        result = ocr.extract_text(empty_plate)
        print(f"   Full text: '{result['full_text']}'")
        print(f"   Detections: {len(result['text'])}")
        print("   ✅ Handled gracefully")
except Exception as e:
    print(f"   ❌ Error: {e}")

# Test 4: Check GPU availability
print("\nTest 4: GPU/CPU Status")
try:
    import torch
    print(f"   PyTorch available: ✅")
    print(f"   CUDA available: {'✅' if torch.cuda.is_available() else '❌'}")
    if torch.cuda.is_available():
        print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
except:
    print(f"   PyTorch not available")

print("\n✅ Error handling tests completed")

## Step 9: Performance Benchmarking

Measure and compare performance metrics

In [ ]:
import time

print("⏱️  Running Performance Benchmarks\n")

if detector and ocr and test_image is not None:
    # Benchmark 1: Detection speed
    num_iterations = 3
    detection_times = []
    
    print(f"Detection Benchmark ({num_iterations} iterations):")
    for i in range(num_iterations):
        start = time.time()
        result = detector.detect(test_image)
        elapsed = time.time() - start
        detection_times.append(elapsed)
        print(f"  Iteration {i+1}: {elapsed*1000:.2f}ms")
    
    avg_detection_time = np.mean(detection_times)
    print(f"  Average: {avg_detection_time*1000:.2f}ms")
    print(f"  Throughput: {1/avg_detection_time:.1f} FPS\n")
    
    # Benchmark 2: OCR speed
    if 'plate_regions' in locals() and plate_regions:
        ocr_times = []
        print(f"OCR Benchmark ({len(plate_regions)} plates):")
        
        for idx, (plate_img, _) in enumerate(plate_regions):
            start = time.time()
            ocr_result = ocr.extract_text(plate_img)
            elapsed = time.time() - start
            ocr_times.append(elapsed)
            print(f"  Plate {idx+1}: {elapsed*1000:.2f}ms")
        
        if ocr_times:
            avg_ocr_time = np.mean(ocr_times)
            print(f"  Average: {avg_ocr_time*1000:.2f}ms per plate\n")
    
    # Benchmark 3: Total pipeline
    print(f"Full Pipeline Benchmark:")
    start = time.time()
    detections = detector.detect(test_image)
    plate_regions = detector.get_plate_regions(test_image, detections)
    total_ocr_time = sum(time.time() for _ in [None])  # Reset
    for plate_img, _ in plate_regions:
        ocr.extract_text(plate_img)
    total_time = time.time() - start
    
    print(f"  Total time: {total_time*1000:.2f}ms")
    print(f"  Throughput: {1/total_time:.1f} FPS")
    
else:
    print("❌ Cannot benchmark: missing detector, OCR, or test image")

## Step 10: Quick Reference - How to Check & Visualize

### For Testing Your Code:

1. **Check Module Loading**
   ```python
   from plate_detector import PlateDetector
   from ocr import PlateOCR
   ```

2. **Load an Image**
   ```python
   import cv2
   image = cv2.imread("path/to/image.jpg")
   ```

3. **Run Detection**
   ```python
   detector = PlateDetector(model_path="...", confidence_threshold=0.5)
   detections = detector.detect(image)
   print(f"Found {detections['num_detections']} plates")
   ```

4. **Visualize Results**
   ```python
   import matplotlib.pyplot as plt
   import cv2
   
   viz = detector.visualize(image, detections)
   viz_rgb = cv2.cvtColor(viz, cv2.COLOR_BGR2RGB)
   plt.imshow(viz_rgb)
   plt.show()
   ```

5. **Run OCR**
   ```python
   ocr = PlateOCR()
   plates = detector.get_plate_regions(image, detections)
   for plate_img, coords in plates:
       result = ocr.extract_text(plate_img)
       print(f"Text: {result['cleaned_text']}")
   ```

### Common Issues & Solutions:

| Issue | Solution |
|-------|----------|
| `ModuleNotFoundError: No module named 'plate_detector'` | Add `sys.path.insert(0, 'src')` before import |
| `Model not found at path` | Update MODEL_PATH to correct path |
| No detections found | Lower `confidence_threshold` or check image quality |
| OCR returns empty text | Ensure detected plate region is large enough |
| GPU out of memory | Set `use_gpu=False` in PlateOCR() |
| Slow performance | Use CPU instead of GPU for testing |

### Keyboard Shortcuts in VS Code:

- **Run Cell**: `Ctrl+Enter` or `Shift+Enter`
- **Clear Cell Output**: Click clear output button
- **View Variable**: Hover over variable or use Variable Inspector
- **Debug Output**: Check Python Terminal at bottom

## Step 11: Additional Testing Tools & Utilities

In [ ]:
# Utility functions for detailed inspection

def inspect_detection_results(detections: dict):
    """Print detailed inspection of detection results"""
    print("📋 Detection Results Inspection")
    print(f"{'='*50}")
    print(f"Total detections: {detections['num_detections']}")
    print(f"Detection boxes shape: {detections['detection_boxes'].shape}")
    print(f"Scores shape: {detections['detection_scores'].shape}")
    
    if detections['num_detections'] > 0:
        print(f"\nDetailed breakdown:")
        for i, (box, score, class_id) in enumerate(zip(
            detections['detection_boxes'],
            detections['detection_scores'],
            detections['detection_classes']
        )):
            ymin, xmin, ymax, xmax = box
            print(f"\n  Detection {i+1}:")
            print(f"    Normalized coordinates: ymin={ymin:.3f}, xmin={xmin:.3f}, ymax={ymax:.3f}, xmax={xmax:.3f}")
            print(f"    Confidence score: {score:.3f}")
            print(f"    Class ID: {int(class_id)}")

def inspect_ocr_results(ocr_result: dict):
    """Print detailed inspection of OCR results"""
    print("📋 OCR Results Inspection")
    print(f"{'='*50}")
    print(f"Raw text: '{ocr_result['full_text']}'")
    print(f"Cleaned text: '{ocr_result['cleaned_text']}'")
    print(f"Average confidence: {ocr_result['avg_confidence']:.3f}")
    print(f"Is valid plate: {ocr_result['is_valid_plate']}")
    print(f"Number of characters detected: {len(ocr_result['text'])}")
    
    if ocr_result['text']:
        print(f"\nCharacter-by-character breakdown:")
        for idx, (char, confidence) in enumerate(zip(ocr_result['text'], ocr_result['confidence'])):
            status = "✓" if confidence > 0.7 else "!" if confidence > 0.4 else "✗"
            print(f"  {idx+1}. '{char}' - {confidence:.3f} {status}")

def compare_images_side_by_side(img1: np.ndarray, img2: np.ndarray, title1: str, title2: str):
    """Display two images side by side for comparison"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB) if len(img1.shape) == 3 and img1.shape[2] == 3 else img1
    img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB) if len(img2.shape) == 3 and img2.shape[2] == 3 else img2
    
    ax1.imshow(img1_rgb)
    ax1.set_title(title1, fontweight='bold', fontsize=12)
    ax1.axis('off')
    
    ax2.imshow(img2_rgb)
    ax2.set_title(title2, fontweight='bold', fontsize=12)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()

def create_detection_report(image: np.ndarray, detections: dict, ocr_results: list = None):
    """Create a comprehensive report of detection and OCR results"""
    print("\n" + "="*60)
    print("PLATE DETECTION & OCR REPORT")
    print("="*60)
    
    print(f"\n📸 Image Information:")
    print(f"  Size: {image.shape}")
    print(f"  Format: {'Color' if len(image.shape) == 3 else 'Grayscale'}")
    
    print(f"\n🔍 Detection Results:")
    print(f"  Plates detected: {detections['num_detections']}")
    
    if detections['num_detections'] > 0:
        scores = detections['detection_scores']
        print(f"  Confidence - Min: {scores.min():.3f}, Max: {scores.max():.3f}, Avg: {scores.mean():.3f}")
    
    if ocr_results:
        print(f"\n📖 OCR Results:")
        valid_count = sum(1 for r in ocr_results if r['is_valid_plate'])
        print(f"  Valid plates: {valid_count}/{len(ocr_results)}")
        
        print(f"\n  Extracted Plate Numbers:")
        for i, result in enumerate(ocr_results, 1):
            text = result['cleaned_text'] if result['cleaned_text'] else result['full_text']
            validity = "✓" if result['is_valid_plate'] else "✗"
            confidence = result['avg_confidence']
            print(f"    {i}. {text:20} | Confidence: {confidence:.3f} | Valid: {validity}")
    
    print("\n" + "="*60)

# Usage examples (uncomment to use):
# if 'detections' in locals():
#     inspect_detection_results(detections)
# if 'all_results' in locals():
#     inspect_ocr_results(all_results[0])
# if 'test_image' in locals() and 'viz_image' in locals():
#     compare_images_side_by_side(test_image, viz_image, "Original", "With Detections")

print("✅ Inspection utilities loaded")

## 📚 Testing & Visualization Guide Summary

### What You Can Do Now:

✅ **Load and Test Modules**
- Import your custom plate detector and OCR modules
- Check for errors during initialization
- Verify GPU/CPU availability

✅ **Run Detection**
- Detect license plates in images
- Visualize detection results with bounding boxes
- Measure performance metrics

✅ **Run OCR**
- Extract text from detected plates
- Clean and validate extracted plate numbers
- View character-level confidence scores

✅ **Visualize Results**
- Display original images
- Show detection boxes with confidence scores
- Show OCR results with text overlays
- Compare original vs processed images

✅ **Debug & Inspect**
- Test error handling
- Inspect detection details
- Create detailed reports
- Benchmark performance

### Files in Your Project:

- `plate_detector.py` - Main detection module
- `ocr.py` - OCR extraction module
- `Test_and_Visualize.ipynb` - This testing notebook
- `requirements.txt` - Dependencies

### Next Steps:

1. **Get a trained model** - Download or train a TensorFlow object detection model
2. **Update MODEL_PATH** - Set the path to your model in the notebook
3. **Test with real images** - Load your own images and test
4. **Monitor performance** - Use the benchmarking tools
5. **Iterate & improve** - Adjust thresholds based on results

### Troubleshooting Quick Links:

- Need to install dependencies? Run: `pip install -r requirements.txt`
- Model loading issues? Check file path and SavedModel format
- OCR not working well? Lower confidence threshold or improve preprocessing
- Memory issues? Use CPU instead of GPU

Good luck with your plate detection project! 🚗🚙🚕